In [1]:
import os
import json
import time
from datetime import datetime, timedelta

from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from zoneinfo import ZoneInfo

# Trading Client
from alpaca.trading.client import (
    TradingClient)                  # Instantiates a client for managing trading and personal brokerage accounts

# Historical Data Clients
from alpaca.data.historical.stock import (
    StockHistoricalDataClient)      # Instantiates a historical data client
from alpaca.data.historical.option import (
    OptionHistoricalDataClient)     # The REST client for interacting with Alpaca Market Data API option data endpoints.

from alpaca.data.timeframe import (
    TimeFrame,                      # A time interval specified in multiples of defined units (minute, day, etc)  # noqa: F401
    TimeFrameUnit)                  # Quantity of time used as unit. Minute | Hour | Day | Week | Month  # noqa: F401

from alpaca.trading.stream import (
    TradingStream)                  # This is a WebSocket client which allows you to streaming data from your trading account.
from alpaca.data.live.option import (
    OptionDataStream)               # A WebSocket client for streaming live option data.

# Data request models
from alpaca.data.requests import (
    StockBarsRequest,               # The request model for retrieving bar data for equities.  # noqa: F401
    StockLatestTradeRequest)        # This request class is used to submit a request for the latest stock trade data.
from alpaca.data.requests import (
    OptionBarsRequest,              # The request model for retrieving bar data for option contracts.  # noqa: F401
    OptionTradesRequest,            # This request class is used to submit a request for option trade data.  # noqa: F401
    OptionLatestQuoteRequest,       # This request class is used to submit a request for the latest option quote data.
    OptionLatestTradeRequest,       # This request class is used to submit a request for the latest option trade data.
    OptionSnapshotRequest,          # This request class is used to submit a request for snapshot data for options.  # noqa: F401
    OptionChainRequest              # This request class is used to submit a request for option chain data for options.
)

# Trading Request Models
from alpaca.trading.requests import (
    GetOptionContractsRequest,      # The parameters that option contracts can be queried by.
    GetAssetsRequest,               # When querying for available assets, this model provides the parameters that can be filtered by.
    LimitOrderRequest,              # Used to submit a limit order.  
    MarketOrderRequest,             # Used to submit a market order.
    GetOrdersRequest,               # Contains data for submitting a request to retrieve orders.   
    ClosePositionRequest,
    OptionLegRequest                # Used for providing details for a leg of a multi-leg order.
)
from alpaca.trading.enums import (
    AssetStatus,                    # Represents the various states for an Asset’s lifecycle. ACTIVE | INACTIVE
    ExerciseStyle,                  # AMERICAN | EUROPEAN
    OrderSide,                      # Represents what side this order was executed on. BUY | SELL
    OrderClass,                     # Represents what class of order this is. SIMPLE | MLEG | BRACKET | OCO | OTO
                                    # The order classes supported by Alpaca vary based on the order’s security type. 
                                    # The following provides a comprehensive breakdown of the supported order classes for each category:
                                    # - Equity trading: simple (or “”), oco, oto, bracket.
                                    # - Options trading: simple (or “”), mleg (required for multi-leg complex options strategies).
                                    # - Crypto trading: simple (or “”).
    OrderStatus,                    # Represents the various states an order can be in. NEW | PARTIALLY_FILLED | FILLED | DONE_FOR_DAY |   # noqa: F401
                                    # CANCELLED | EXPIRED | REPLACED | PENDING_CANCEL | PENDING_REPLACE | PENDING_REVIEW | PENDING_NEW |
                                    # ACCEPTED | ACCEPTED_FOR_BIDDING | STOPPED | REJECTED | SUSPENDED | CALCULATED | HELD
    OrderType,                      # Represents the type of order this is. MARKET | LIMIT | STOP | STOP_LIMIT | TRAILING_STOP      
                                    # The order types supported by Alpaca vary based on the order’s security type. 
                                    # The following provides a comprehensive breakdown of the supported order types for each category:
                                    # - Equity trading: market, limit, stop, stop_limit, trailing_stop.
                                    # - Options trading: market, limit, stop, stop_limit.
                                    # - Crypto trading: market, limit, stop_limit.
    TimeInForce,                    # Represents the various time in force options for an Order. DAY | GTC | OPG | CLS | IOC | FOK
                                    # The Time-In-Force values supported by Alpaca vary based on the order’s security type.
                                    # Here is a breakdown of the supported TIFs for each specific security type:
                                    # - Equity trading: day, gtc, opg, cls, ioc, fok.
                                    # - Options trading: day.
                                    # - Crypto trading: gtc, ioc
    QueryOrderStatus,               # OPEN | CLOSED | ALL
    ContractType                    # CALL | PUT
)
from alpaca.common.exceptions import APIError

In [2]:
# Please safely store your API keys and never commit them to the repository (use .gitignore)
# Load environment variables from environment file (e.g. .env)
load_dotenv()

TRADE_API_KEY = os.environ.get('ALPACA_API_KEY')
TRADE_API_SECRET = os.environ.get('ALPACA_SECRET_KEY')

# We use paper environment for this example
ALPACA_PAPER_TRADE = True

# Below are the variables for development this documents so please do not change these variables
TRADE_API_URL = "https://paper-api.alpaca.markets" if ALPACA_PAPER_TRADE else "https://api.alpaca.markets"
DATA_API_URL = None if ALPACA_PAPER_TRADE else "https://data.alpaca.markets"
TRADE_API_WSS = None
OPTION_STREAM_DATA_WSS = None

In [3]:
# To enable live trading, replace `ALPACA_PAPER_TRADE = True` with `ALPACA_PAPER_TRADE = False` to disable the paper trading configuration
trade_client = TradingClient(api_key=TRADE_API_KEY, secret_key=TRADE_API_SECRET, paper=ALPACA_PAPER_TRADE)
stock_data_client = StockHistoricalDataClient(api_key=TRADE_API_KEY, secret_key=TRADE_API_SECRET)

In [4]:
# Check account related information
acc = trade_client.get_account()
acc

{   'account_blocked': False,
    'account_number': 'PA3ZPB2W362P',
    'accrued_fees': '0',
    'buying_power': '136873.4',
    'cash': '34267.2',
    'created_at': datetime.datetime(2025, 2, 7, 1, 47, 10, 815394, tzinfo=TzInfo(UTC)),
    'crypto_status': <AccountStatus.ACTIVE: 'ACTIVE'>,
    'currency': 'USD',
    'daytrade_count': 0,
    'daytrading_buying_power': '0',
    'equity': '102606.2',
    'id': UUID('94ccd725-6370-4308-b880-31e14648310a'),
    'initial_margin': '34169.5',
    'last_equity': '102606.2',
    'last_maintenance_margin': '20501.7',
    'long_market_value': '68339',
    'maintenance_margin': '20501.7',
    'multiplier': '2',
    'non_marginable_buying_power': '66436.7',
    'options_approved_level': 3,
    'options_buying_power': '68436.7',
    'options_trading_level': 3,
    'pattern_day_trader': False,
    'pending_transfer_in': None,
    'pending_transfer_out': None,
    'portfolio_value': '102606.2',
    'regt_buying_power': '136873.4',
    'short_market_val

In [5]:
acc.cash

'34267.2'

In [6]:
# Check account configurations
acct_config = trade_client.get_account_configurations()
acct_config

{   'dtbp_check': <DTBPCheck.ENTRY: 'entry'>,
    'fractional_trading': True,
    'max_margin_multiplier': '4',
    'max_options_trading_level': None,
    'no_shorting': False,
    'pdt_check': <PDTCheck.ENTRY: 'entry'>,
    'ptp_no_exception_entry': False,
    'suspend_trade': False,
    'trade_confirm_email': <TradeConfirmationEmail.ALL: 'all'>}

In [7]:
# We can filter assets by `options_enabled` attribute
# Asset object has `options_enabled` attribute if it is options enabled
req = GetAssetsRequest(
  attributes = "options_enabled"  
)
assets = trade_client.get_all_assets(req)
# asseys is a list of Asset models (classes) where each asset model is for a symbol
assets

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': ['has_options'],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.OTC: 'OTC'>,
     'fractionable': False,
     'id': UUID('3642a73c-3c51-4681-87bc-008078218aee'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min_trade_increment': None,
     'name': 'Scorpius Holdings, Inc.',
     'price_increment': None,
     'shortable': False,
     'status': <AssetStatus.ACTIVE: 'active'>,
     'symbol': 'SCPX',
     'tradable': False},
 {   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'attributes': ['fractional_eh_enabled', 'has_options'],
     'easy_to_borrow': False,
     'exchange': <AssetExchange.OTC: 'OTC'>,
     'fractionable': True,
     'id': UUID('b0114538-78e0-4ee9-bce6-646a16f6c993'),
     'maintenance_margin_requirement': 100.0,
     'marginable': False,
     'min_order_size': None,
     'min_trade_increment': None,
     'name

In [8]:
assets[0].symbol

'SCPX'

In [9]:
# In this instance, we retrieve the necessary contracts based on the parameters we've configured.
# specify underlying symbol

underlying_symbol = ["AAPL","SPY"]
req = GetOptionContractsRequest(
    underlying_symbols=underlying_symbol,  
    # specify asset status: active (default)
    status=AssetStatus.ACTIVE,   
    # specify expiration date (specified date + 1 day range)
    expiration_date = None,
    # we can pass a date object or string (YYYY-MM-DD) to specify an expiration date range
    expiration_date_gte = None,                            
    expiration_date_lte = None, 
    # specify root symbol
    root_symbol = None,
    # specify option type (ContractType.CALL or ContractType.PUT)
    type = None,  
)

res = trade_client.get_option_contracts(req)
res

{   'next_page_token': 'MTAw',
    'option_contracts': [   {   'close_price': '168.32',
                                'close_price_date': datetime.date(2025, 11, 26),
                                'expiration_date': datetime.date(2025, 12, 5),
                                'id': '4ddfbab9-ea0d-4b3c-878d-d6861c77d3a6',
                                'name': 'AAPL Dec 05 2025 110 Call',
                                'open_interest': '7',
                                'open_interest_date': datetime.date(2025, 11, 26),
                                'root_symbol': 'AAPL',
                                'size': '100',
                                'status': <AssetStatus.ACTIVE: 'active'>,
                                'strike_price': 110.0,
                                'style': <ExerciseStyle.AMERICAN: 'american'>,
                                'symbol': 'AAPL251205C00110000',
                                'tradable': True,
                                'type': <Co

In [10]:
# Get option contract by option symbol
symbol = res.option_contracts[0].symbol
contract = trade_client.get_option_contract(symbol)
contract

{   'close_price': '168.32',
    'close_price_date': datetime.date(2025, 11, 26),
    'expiration_date': datetime.date(2025, 12, 5),
    'id': '4ddfbab9-ea0d-4b3c-878d-d6861c77d3a6',
    'name': 'AAPL Dec 05 2025 110 Call',
    'open_interest': '7',
    'open_interest_date': datetime.date(2025, 11, 26),
    'root_symbol': 'AAPL',
    'size': '100',
    'status': <AssetStatus.ACTIVE: 'active'>,
    'strike_price': 110.0,
    'style': <ExerciseStyle.AMERICAN: 'american'>,
    'symbol': 'AAPL251205C00110000',
    'tradable': True,
    'type': <ContractType.CALL: 'call'>,
    'underlying_asset_id': UUID('b0b6dd9d-8b9b-48a9-ba46-b9d54906e415'),
    'underlying_symbol': 'AAPL'}

In [11]:
# or we can get options contract by id
contract_id = res.option_contracts[0].id
contract = trade_client.get_option_contract(symbol_or_id=contract_id)
contract

{   'close_price': '168.32',
    'close_price_date': datetime.date(2025, 11, 26),
    'expiration_date': datetime.date(2025, 12, 5),
    'id': '4ddfbab9-ea0d-4b3c-878d-d6861c77d3a6',
    'name': 'AAPL Dec 05 2025 110 Call',
    'open_interest': '7',
    'open_interest_date': datetime.date(2025, 11, 26),
    'root_symbol': 'AAPL',
    'size': '100',
    'status': <AssetStatus.ACTIVE: 'active'>,
    'strike_price': 110.0,
    'style': <ExerciseStyle.AMERICAN: 'american'>,
    'symbol': 'AAPL251205C00110000',
    'tradable': True,
    'type': <ContractType.CALL: 'call'>,
    'underlying_asset_id': UUID('b0b6dd9d-8b9b-48a9-ba46-b9d54906e415'),
    'underlying_symbol': 'AAPL'}

In [12]:
# Get PUT option contracts
# We can filter the contracts that are relevant to us by specifying various option properties like 
# expiration date range (the contracts which are expiring within a particular time window), 
# strike range (the contracts with strike prices within a given range), style of expiration for 
# the option contract (like American or European), etc.

underlying_symbol = "SPY"

# specify expiration date range
now = datetime.now(tz=ZoneInfo("America/New_York"))
day1 = now + timedelta(days=1)
day60 = now + timedelta(days=60)

req = GetOptionContractsRequest(
    # specify underlying symbol
    underlying_symbols = [underlying_symbol],
    # specify asset status: active (default)
    status = AssetStatus.ACTIVE,
    # specify expiration date (specified date + 1 day range)
    expiration_date = None,
    # we can pass date object or string
    expiration_date_gte = day1.date(),
    expiration_date_lte = day60.strftime(format="%Y-%m-%d"),
    # specify root symbol
    root_symbol = None,
    # specify option type: put
    type = ContractType.PUT,    
    # specify option style: american
    style = ExerciseStyle.AMERICAN,
    # specify strike price range
    strike_price_gte = None,
    strike_price_lte = None,
    # specify limit
    limit = 100,
    # specify page
    page_token = None
)

res = trade_client.get_option_contracts(req)
res

{   'next_page_token': 'MTAw',
    'option_contracts': [   {   'close_price': '0.01',
                                'close_price_date': datetime.date(2025, 11, 25),
                                'expiration_date': datetime.date(2025, 12, 1),
                                'id': '8c775d00-984c-4a82-8269-7d20f3374a94',
                                'name': 'SPY Dec 01 2025 500 Put',
                                'open_interest': '373',
                                'open_interest_date': datetime.date(2025, 11, 26),
                                'root_symbol': 'SPY',
                                'size': '100',
                                'status': <AssetStatus.ACTIVE: 'active'>,
                                'strike_price': 500.0,
                                'style': <ExerciseStyle.AMERICAN: 'american'>,
                                'symbol': 'SPY251201P00500000',
                                'tradable': True,
                                'type': <Contra

In [13]:
# Filter contract with the highest open interest

max_open_interest = 0
high_open_interest_contract = None
for contract in res.option_contracts:
    oi = contract.open_interest
    if oi is not None:
        try:
            oi_int = int(oi)
            if oi_int > max_open_interest:
                max_open_interest = oi_int
                high_open_interest_contract = contract
        # skip if not convertible to int
        except ValueError:
            continue 

In [ ]:
def is_trading_window() -> bool:
    current_time_ny = datetime.now(tz=ZoneInfo("America/New_York"))
    is_weekday = current_time_ny.weekday() < 5  # Monday=0, Sunday=6
    is_after_930am = current_time_ny.hour > 10 or (current_time_ny.hour == 9 and current_time_ny.minute >= 30)
    is_before_4pm = current_time_ny.hour < 16
    # Use US holiday API or static list for demonstration
    us_holidays = [
        "2025-01-01", "2025-01-20", "2025-02-17", "2025-04-18", "2025-05-26",
        "2025-19-06", "2025-07-04", "2025-09-01", "2025-11-27", "2025-12-25"
    ]
    date_str = current_time_ny.strftime("%Y-%m-%d")
    is_us_holiday = date_str in us_holidays

    is_trading_window = is_weekday and is_after_930am and is_before_4pm  and not is_us_holiday

    return is_trading_window

print(f"Is trading window open? {is_trading_window()}")

Is trading window open? False


In [15]:
# Check contract information like expiry, expiration style and contract status for the higest open interest option
if high_open_interest_contract is not None:
    #Check option expiry
    print(high_open_interest_contract.expiration_date)

    #Check option status
    print(high_open_interest_contract.status)

    #Check option exercise style
    print(high_open_interest_contract.style)

2025-12-01
AssetStatus.ACTIVE
ExerciseStyle.AMERICAN


In [51]:
# Execute either a Buy Call or Put option order in accordance with your trading strategy
# On expiration days, option orders must be submitted before 3:15 p.m. ET and before 3:30 p.m. ET 
# for broad-based ETFs like SPY and QQQ. Orders placed after these times will be rejected.
# Expiring positions are auto-liquidated starting at 3:30 p.m. ET (options) and 3:45 p.m. ET 
# (broad-based ETFs) for risk management.

if high_open_interest_contract is None:
    raise ValueError("No high open interest contract found.")
if is_trading_window():
    req = MarketOrderRequest(
        symbol=high_open_interest_contract.symbol, # type: ignore
        qty=1,
        side=OrderSide.BUY,
        type=OrderType.MARKET,
        time_in_force=TimeInForce.DAY,
    )
else:
    req = LimitOrderRequest(
        symbol=high_open_interest_contract.symbol, # type: ignore
        qty=1,
        side=OrderSide.BUY,
        type=OrderType.LIMIT,
        time_in_force=TimeInForce.DAY,
        limit_price=round(float(high_open_interest_contract.close_price) * 1.01,2), # type: ignore
    )
print(req)
res = trade_client.submit_order(req)
res

symbol='SPY250818P00640000' qty=1.0 notional=None side=<OrderSide.BUY: 'buy'> type=<OrderType.LIMIT: 'limit'> time_in_force=<TimeInForce.DAY: 'day'> order_class=None extended_hours=None client_order_id=None legs=None take_profit=None stop_loss=None position_intent=None limit_price=0.7


{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
    'asset_id': UUID('7a6e7770-a96b-4250-adc0-597b6e0448a2'),
    'canceled_at': None,
    'client_order_id': 'ecddf4b0-8d69-4561-a0ca-20f44a04f012',
    'created_at': datetime.datetime(2025, 8, 16, 23, 22, 18, 277063, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2025, 8, 18, 20, 15, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('fe5f4dca-3748-4615-9c38-b384d864b1e4'),
    'legs': None,
    'limit_price': '0.7',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStat

In [52]:
# Get list of requested orders by specifying option contract symbol
req = GetOrdersRequest(
    status=QueryOrderStatus.ALL,
    symbols=[high_open_interest_contract.symbol], # type: ignore
    limit=2,
)
orders = trade_client.get_orders(req)
orders

[{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
     'asset_id': UUID('7a6e7770-a96b-4250-adc0-597b6e0448a2'),
     'canceled_at': None,
     'client_order_id': 'ecddf4b0-8d69-4561-a0ca-20f44a04f012',
     'created_at': datetime.datetime(2025, 8, 16, 23, 22, 18, 277063, tzinfo=TzInfo(UTC)),
     'expired_at': None,
     'expires_at': datetime.datetime(2025, 8, 18, 20, 15, tzinfo=TzInfo(UTC)),
     'extended_hours': False,
     'failed_at': None,
     'filled_at': None,
     'filled_avg_price': None,
     'filled_qty': '0',
     'hwm': None,
     'id': UUID('fe5f4dca-3748-4615-9c38-b384d864b1e4'),
     'legs': None,
     'limit_price': '0.7',
     'notional': None,
     'order_class': <OrderClass.SIMPLE: 'simple'>,
     'order_type': <OrderType.LIMIT: 'limit'>,
     'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
     'qty': '1',
     'ratio_qty': None,
     'replaced_at': None,
     'replaced_by': None,
     'replaces': None,
     'side': <OrderSide.BUY: 'buy'>

In [53]:
# We can specify a client_order_id when placing an order
import uuid
client_order_id = f"order_{uuid.uuid4().hex[:8]}"

# Execute a Buy Call or Put option order with a custom client_order_id for order tracking and identification
if high_open_interest_contract is None:
    raise ValueError("No high open interest contract found.")
if is_trading_window():
    req = MarketOrderRequest(
        symbol=high_open_interest_contract.symbol, # type: ignore
        qty=1,
        side=OrderSide.BUY,
        type=OrderType.MARKET,
        time_in_force=TimeInForce.DAY,
        client_order_id=client_order_id,        
    )
else:
    req = LimitOrderRequest(
        symbol=high_open_interest_contract.symbol, # type: ignore
        qty=1,
        side=OrderSide.BUY,
        type=OrderType.LIMIT,
        time_in_force=TimeInForce.DAY,
        limit_price=round(float(high_open_interest_contract.close_price) * 1.01,2), # type: ignore
        client_order_id=client_order_id,
    )
print(req)
res = trade_client.submit_order(req)
res

symbol='SPY250818P00640000' qty=1.0 notional=None side=<OrderSide.BUY: 'buy'> type=<OrderType.LIMIT: 'limit'> time_in_force=<TimeInForce.DAY: 'day'> order_class=None extended_hours=None client_order_id='order_e8207c59' legs=None take_profit=None stop_loss=None position_intent=None limit_price=0.7


{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
    'asset_id': UUID('7a6e7770-a96b-4250-adc0-597b6e0448a2'),
    'canceled_at': None,
    'client_order_id': 'order_e8207c59',
    'created_at': datetime.datetime(2025, 8, 16, 23, 44, 14, 651589, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2025, 8, 18, 20, 15, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('8e9c7505-a30c-419c-a272-cf675007ef1d'),
    'legs': None,
    'limit_price': '0.7',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.ACCEPTED: 'accepted

In [56]:
# and filter the order with the desired client_order_id
print(f"client_order_id: {client_order_id}")
# Extract order information by the custom client_order_id
trade_client.get_order_by_client_id(client_id=client_order_id)

client_order_id: order_e8207c59


{   'asset_class': <AssetClass.US_OPTION: 'us_option'>,
    'asset_id': UUID('7a6e7770-a96b-4250-adc0-597b6e0448a2'),
    'canceled_at': None,
    'client_order_id': 'order_e8207c59',
    'created_at': datetime.datetime(2025, 8, 16, 23, 44, 14, 651589, tzinfo=TzInfo(UTC)),
    'expired_at': None,
    'expires_at': datetime.datetime(2025, 8, 18, 20, 15, tzinfo=TzInfo(UTC)),
    'extended_hours': False,
    'failed_at': None,
    'filled_at': None,
    'filled_avg_price': None,
    'filled_qty': '0',
    'hwm': None,
    'id': UUID('8e9c7505-a30c-419c-a272-cf675007ef1d'),
    'legs': None,
    'limit_price': '0.7',
    'notional': None,
    'order_class': <OrderClass.SIMPLE: 'simple'>,
    'order_type': <OrderType.LIMIT: 'limit'>,
    'position_intent': <PositionIntent.BUY_TO_OPEN: 'buy_to_open'>,
    'qty': '1',
    'ratio_qty': None,
    'replaced_at': None,
    'replaced_by': None,
    'replaces': None,
    'side': <OrderSide.BUY: 'buy'>,
    'status': <OrderStatus.ACCEPTED: 'accepted

### Positions

In [16]:
# Get all postions 
positions = trade_client.get_all_positions()
positions

[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'asset_marginable': True,
     'avg_entry_price': '648.9',
     'avg_entry_swap_rate': None,
     'change_today': '0',
     'cost_basis': '64890',
     'current_price': '683.39',
     'exchange': <AssetExchange.ARCA: 'ARCA'>,
     'lastday_price': '683.39',
     'market_value': '68339',
     'qty': '100',
     'qty_available': '100',
     'side': <PositionSide.LONG: 'long'>,
     'swap_rate': None,
     'symbol': 'SPY',
     'unrealized_intraday_pl': '0',
     'unrealized_intraday_plpc': '0',
     'unrealized_pl': '3449',
     'unrealized_plpc': '0.0531514871320697',
     'usd': None}]

In [17]:
# Get open positions by symbol (if there are any)
try:
    positions = trade_client.get_open_position(
        symbol_or_asset_id=high_open_interest_contract.symbol
    )
except APIError as e:
    print(f"Error retrieving open position: {e}")
positions

Error retrieving open position: {"code":40410000,"message":"position does not exist"}


[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'asset_marginable': True,
     'avg_entry_price': '648.9',
     'avg_entry_swap_rate': None,
     'change_today': '0',
     'cost_basis': '64890',
     'current_price': '683.39',
     'exchange': <AssetExchange.ARCA: 'ARCA'>,
     'lastday_price': '683.39',
     'market_value': '68339',
     'qty': '100',
     'qty_available': '100',
     'side': <PositionSide.LONG: 'long'>,
     'swap_rate': None,
     'symbol': 'SPY',
     'unrealized_intraday_pl': '0',
     'unrealized_intraday_plpc': '0',
     'unrealized_pl': '3449',
     'unrealized_plpc': '0.0531514871320697',
     'usd': None}]

In [18]:
# Get open positions by contract id
try:
    positions = trade_client.get_open_position(
    symbol_or_asset_id=high_open_interest_contract.id
    )
except APIError as e:
    print(f"Error retrieving open position: {e}")
positions

Error retrieving open position: {"code":40410000,"message":"position does not exist"}


[{   'asset_class': <AssetClass.US_EQUITY: 'us_equity'>,
     'asset_id': UUID('b28f4066-5c6d-479b-a2af-85dc1a8f16fb'),
     'asset_marginable': True,
     'avg_entry_price': '648.9',
     'avg_entry_swap_rate': None,
     'change_today': '0',
     'cost_basis': '64890',
     'current_price': '683.39',
     'exchange': <AssetExchange.ARCA: 'ARCA'>,
     'lastday_price': '683.39',
     'market_value': '68339',
     'qty': '100',
     'qty_available': '100',
     'side': <PositionSide.LONG: 'long'>,
     'swap_rate': None,
     'symbol': 'SPY',
     'unrealized_intraday_pl': '0',
     'unrealized_intraday_plpc': '0',
     'unrealized_pl': '3449',
     'unrealized_plpc': '0.0531514871320697',
     'usd': None}]

In [1]:
# Check the cost basis and unrealised profit/loss of the open position
# Show cost basis
if isinstance(positions, list):
    if len(positions) == 0:
        print("No open positions found.")
    else:
        # Show unrealized profit/loss
        print(positions.unrealized_pl)

NameError: name 'positions' is not defined

In [70]:
# Close the option position
try:
    trade_client.close_position(
        symbol_or_asset_id=high_open_interest_contract.symbol,
        close_options=ClosePositionRequest(qty="1")
    )
except APIError as e:
    print(f"Error closing position: {e}")

Error closing position: {"code":40410000,"message":"position not found: SPY250818P00640000"}


In [71]:
# !pip install nest_asyncio
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# A bit odd. Needs more work.
# Retrieve real-time live trade update data from the moment the order is placed until execution.
# Subscribe trade updates

trade_stream_client = TradingStream(TRADE_API_KEY, TRADE_API_SECRET, paper=TRADE_API_URL, url_override=TRADE_API_WSS)

async def trade_updates_handler(data):
    print(data)

trade_stream_client.subscribe_trade_updates(trade_updates_handler)
trade_stream_client.run()

keyboard interrupt, bye


In [76]:
# Market Data (Historical)
# setup option historical data client
option_historical_data_client = OptionHistoricalDataClient(TRADE_API_KEY, TRADE_API_SECRET, url_override=DATA_API_URL)

# get option latest quote by symbol
req = OptionLatestQuoteRequest(
    symbol_or_symbols=[high_open_interest_contract.symbol],
)
option_historical_data_client.get_option_latest_quote(req)

{'SPY250818P00640000': {   'ask_exchange': 'P',
     'ask_price': 0.67,
     'ask_size': 156.0,
     'bid_exchange': 'U',
     'bid_price': 0.66,
     'bid_size': 7.0,
     'conditions': ' ',
     'symbol': 'SPY250818P00640000',
     'tape': None,
     'timestamp': datetime.datetime(2025, 8, 15, 19, 59, 59, 29857, tzinfo=TzInfo(UTC))}}

In [77]:
# get option latest trade by symbol
req = OptionLatestTradeRequest(
    symbol_or_symbols=[high_open_interest_contract.symbol],
)
option_historical_data_client.get_option_latest_trade(req)

{'SPY250818P00640000': {   'conditions': 'a',
     'exchange': 'I',
     'id': None,
     'price': 0.71,
     'size': 1.0,
     'symbol': 'SPY250818P00640000',
     'tape': None,
     'timestamp': datetime.datetime(2025, 8, 15, 20, 14, 59, 732374, tzinfo=TzInfo(UTC))}}

In [79]:
# get option chain by underlying_symbol
req = OptionChainRequest(
    underlying_symbol=high_open_interest_contract.underlying_symbol,
)
option_historical_data_client.get_option_chain(req)

{'SPY250821P00656000': {   'greeks': {   'delta': -0.9172,
                   'gamma': 0.0175,
                   'rho': -0.0661,
                   'theta': -0.0943,
                   'vega': 0.1028},
     'implied_volatility': 0.1293,
     'latest_quote': {   'ask_exchange': 'I',
                         'ask_price': 12.8,
                         'ask_size': 10.0,
                         'bid_exchange': 'B',
                         'bid_price': 12.35,
                         'bid_size': 1.0,
                         'conditions': ' ',
                         'symbol': 'SPY250821P00656000',
                         'tape': None,
                         'timestamp': datetime.datetime(2025, 8, 15, 19, 59, 59, 65182, tzinfo=TzInfo(UTC))},
     'latest_trade': {   'conditions': 'a',
                         'exchange': 'I',
                         'id': None,
                         'price': 11.94,
                         'size': 1.0,
                         'symbol': 'SPY25082

In [81]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Needs work
# Live streaming of market data based on a collection of symbols or an individual symbol
option_data_stream_client = OptionDataStream(TRADE_API_KEY, TRADE_API_SECRET, url_override=OPTION_STREAM_DATA_WSS)

async def option_data_stream_handler(data):
    print(data)

symbols=[
    high_open_interest_contract.symbol,
]

option_data_stream_client.subscribe_quotes(option_data_stream_handler, *symbols) 
option_data_stream_client.subscribe_trades(option_data_stream_handler, *symbols)

option_data_stream_client.run()

keyboard interrupt, bye


TimeoutError: 

### Place Multi-Leg Options

In [83]:
# Get the underlying stock price and define the strike range to narrow down the option chain.
# Define a underlying symbol we want to trade
underlying_symbol = 'SPY'

# Get the latest price of the underlying stock
def get_underlying_price(symbol):
    underlying_trade_request = StockLatestTradeRequest(symbol_or_symbols=symbol)
    underlying_trade_response = stock_data_client.get_stock_latest_trade(underlying_trade_request)
    return underlying_trade_response[symbol].price

# Get the latest price of the underlying stock
underlying_price = get_underlying_price(underlying_symbol)

# Define a 1% range around the underlying price 
STRIKE_RANGE = 0.01

# Set the minimum and maximum strike prices based on the underlying price
min_strike = str(underlying_price * (1 - STRIKE_RANGE))
max_strike = str(underlying_price * (1 + STRIKE_RANGE))

print(f"{underlying_symbol} price: {underlying_price}")

SPY price: 643.16


In [86]:
# Define a function to find near at-the-money (ATM) options.
# This is a function that will return a contract which minimizes the difference from a target price
def find_nearest_strike_contract(contracts, target_price):
    min_diff = 0
    min_contract = None
    for contract in contracts:
        diff = abs(float(contract.strike_price) - target_price)
        if min_contract is None or diff < min_diff:
            min_diff = diff
            min_contract = contract
    return min_contract

In [95]:
# Narrow down the options to near-ATM options.
# Obtain both call options and put options of the specified underlying asset
underlying_symbol = ['SPY']
order_legs = []

for c_type in [ContractType.CALL, ContractType.PUT]:
    print(c_type)
    req = GetOptionContractsRequest(
        underlying_symbols=underlying_symbol,
        status=AssetStatus.ACTIVE,
        expiration_date_gte = now.date() + timedelta(days=7),
        expiration_date_lte = now.date() + timedelta(days=8),
        style=ExerciseStyle.AMERICAN,
        strike_price_gte=min_strike,
        strike_price_lte=max_strike,
        type=c_type,
        page_token=None
    )
    # Get option chain of the underlying symbol for both contract types
    cts = trade_client.get_option_contracts(req)
    # print(cts)

    c = find_nearest_strike_contract(cts.option_contracts, underlying_price)
    order_legs.append(OptionLegRequest(
        symbol=c.symbol,
        side=OrderSide.BUY,
        ratio_qty=1
    ))

# We should see that the symbols are similar, like "SPY______C________" and "SPY______P________",
# with all values marked as "_" being the same in both symbols.
# Such is because we expect only the contract type (call or put, C or P) to be different.

order_legs

ContractType.CALL
ContractType.PUT


[{   'position_intent': None,
     'ratio_qty': 1.0,
     'side': <OrderSide.BUY: 'buy'>,
     'symbol': 'SPY250825C00644000'},
 {   'position_intent': None,
     'ratio_qty': 1.0,
     'side': <OrderSide.BUY: 'buy'>,
     'symbol': 'SPY250825P00644000'}]

In [ ]:
# Place the order for both legs simultaneously to execute the long straddle.
# Set up an order request
req = MarketOrderRequest(
    qty=1,
    order_class=OrderClass.MLEG,
    time_in_force=TimeInForce.DAY,
    legs=order_legs
)
# place the order of the long straddle
res = trade_client.submit_order(req)
res

### Place Multi-Leg Combo Order Position
You can also submit multi-leg (MLeg) orders that include an equity leg. At this stage, the system supports MLeg equity orders only if the equity leg is combined with a short call option leg—i.e., a covered call strategy. The equity-leg quantity must match the underlying option contract ratio (for example, 1 contract corresponds to 100 shares of the underlying stock). The same requirement applies to exit orders. The total amount for the entire MLeg order is governed by the parent quantity (i.e., the main order size), ensuring consistent scaling across all legs.

In [90]:
# Get the underlying stock price and define the strike range to narrow down the option chain.
# Define a underlying symbol we want to trade
underlying_symbol = 'SPY'

# Get the latest price of the underlying stock
underlying_trade_request = StockLatestTradeRequest(symbol_or_symbols=underlying_symbol)
underlying_trade_response = stock_data_client.get_stock_latest_trade(underlying_trade_request)
underlying_price = underlying_trade_response[underlying_symbol].price

# Define a 1% range around the underlying price 
STRIKE_RANGE = 0.01

# Set the maximum strike price based on the underlying price and set the minimum strike price to the underlying price
min_strike = str(underlying_price)
max_strike = str(underlying_price * (1 + STRIKE_RANGE))

print(f"{underlying_symbol} price: {underlying_price}")

SPY price: 643.16


In [91]:
# Narrow down the options to near-ATM options
# Obtain call options of the specified underlying asset
underlying_symbol = ['SPY']
now = datetime.now(tz=ZoneInfo("America/New_York"))

req = GetOptionContractsRequest(
    underlying_symbols=underlying_symbol,
    status=AssetStatus.ACTIVE,
    expiration_date_gte = now.date() + timedelta(days=13),
    expiration_date_lte = now.date() + timedelta(days=22),
    style=ExerciseStyle.AMERICAN,
    strike_price_gte=min_strike,
    strike_price_lte=max_strike,
    type=ContractType.CALL,
    page_token=None
)
# Get option chain of the underlying symbol for both contract types
cts = trade_client.get_option_contracts(req)
cts

{   'next_page_token': None,
    'option_contracts': [   {   'close_price': None,
                                'close_price_date': None,
                                'expiration_date': datetime.date(2025, 9, 2),
                                'id': '97f2ebe8-dad9-44ad-a76e-2ef75a7adfe0',
                                'name': 'SPY Sep 02 2025 645 Call',
                                'open_interest': None,
                                'open_interest_date': None,
                                'root_symbol': 'SPY',
                                'size': '100',
                                'status': <AssetStatus.ACTIVE: 'active'>,
                                'strike_price': 645.0,
                                'style': <ExerciseStyle.AMERICAN: 'american'>,
                                'symbol': 'SPY250902C00645000',
                                'tradable': True,
                                'type': <ContractType.CALL: 'call'>,
                             

In [92]:
# Get the option contract symbol of the 2nd call option in the option chain we obtained in the previous step
cts.option_contracts[2].symbol

'SPY250905C00645000'

In [93]:
# Create a list for the combo order request (short call + 100 shares of the underlying security)
order_legs = []
order_legs.append(OptionLegRequest(
    symbol=cts.option_contracts[2].symbol,
    side=OrderSide.SELL,
    ratio_qty=1
))

order_legs.append(OptionLegRequest(
    symbol='SPY',
    side=OrderSide.BUY,
    ratio_qty=100
))

In [94]:
# Place the order for both legs simultaneously to execute the combo order
# Set up an order request
req = MarketOrderRequest(
    qty=1,
    order_class=OrderClass.MLEG,
    time_in_force=TimeInForce.DAY,
    legs=order_legs
)
# place the order of the combo order (short call + 100 shares of the underlying security)
res = trade_client.submit_order(req)
res

APIError: {"code":42210000,"message":"options market orders are only allowed during market hours"}